In [1]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Note: no special encoder import needed here — Frequency Encoding isn't a
# built-in scikit-learn transformer, we implement it manually using pandas
# (each category is replaced by how often it appears in the TRAINING data)

In [2]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds — same folds as every
# other notebook, for valid paired comparison later
folds = get_cv_folds(df)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 5 folds


In [3]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set
X = df.drop(columns=["advertised_price"])

# Categorical columns to encode
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

# Numeric columns — vehicle_age_days excluded (redundant with vehicle_age_years,
# confirmed via feature importance comparison in the One-Hot notebook)
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_years"]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: 8
Numeric columns: 7


In [4]:
# Reset results dictionary — storing RMSE, MAE, and R² for each model, per fold
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

In [5]:
# Loop through each of the 5 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- Frequency Encode the categorical columns ----
    # For each categorical column, replace each category with how many times
    # it appears in the TRAINING data. No scikit-learn transformer for this,
    # so we build the mapping manually using value_counts().
    X_train_cat = pd.DataFrame(index=X_train.index)
    X_test_cat = pd.DataFrame(index=X_test.index)

    for col in categorical_cols:
        # Count how often each category appears, using ONLY training data
        # (this is the leakage-prevention step — test data must never
        # contribute to these counts)
        freq_map = X_train[col].value_counts()

        # Apply this mapping to both train and test.
        # .map() looks up each row's category in freq_map and replaces it
        # with the count. If a category in the test set was never seen in
        # training, .map() returns NaN — we fill those with 0, meaning
        # "this category was effectively unseen/rare from the model's
        # perspective."
        X_train_cat[col] = X_train[col].map(freq_map).fillna(0)
        X_test_cat[col] = X_test[col].map(freq_map).fillna(0)

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
        ],
        final_estimator=LinearRegression(),
        cv=3,
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))

    # Stacking (unscaled features)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    # Print progress so we can see the loop working fold by fold
    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

Fold 1/5 complete
Fold 2/5 complete
Fold 3/5 complete
Fold 4/5 complete
Fold 5/5 complete

All folds complete.


In [6]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the folds, per model
summary_rows = []
for model_name, metrics in results.items():
    summary_rows.append({
        "model": model_name,
        "mean_rmse": np.mean(metrics["rmse"]),
        "std_rmse": np.std(metrics["rmse"]),
        "mean_mae": np.mean(metrics["mae"]),
        "std_mae": np.std(metrics["mae"]),
        "mean_r2": np.mean(metrics["r2"]),
        "std_r2": np.std(metrics["r2"])
    })

summary = pd.DataFrame(summary_rows)
summary

,model,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2
0,Linear Regression,5086.368823,133.568980,3054.737221,28.419179,0.869162,0.003830
1,KNN,2668.406880,107.823533,1483.884613,19.005456,0.963981,0.002060
2,Decision Tree,3058.041962,121.475766,1646.806519,21.943945,0.952694,0.002650
3,Random Forest,2151.799346,122.055866,1179.865522,20.218066,0.976563,0.002051
4,Stacking,2163.329195,120.396574,1189.226549,20.711492,0.976313,0.002028


In [7]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
# Same structure as every other encoder notebook, so all files can be combined later
rows = []
for model_name, metrics in results.items():
    for fold_i in range(len(folds)):
        rows.append({
            "encoder": "frequency",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("frequency_scores.csv", index=False)

print("Saved results to frequency_scores.csv")

Saved results to frequency_scores.csv


In [8]:
# Capture feature importance from Random Forest, same as previous notebooks
feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance["encoder"] = "frequency"
feature_importance.to_csv("frequency_feature_importance.csv", index=False)

feature_importance.head(15)

,feature,importance,encoder
1,list_price,0.671069,frequency
6,vehicle_age_years,0.160119,frequency
0,mileage,0.099801,frequency
2,reg_year,0.013302,frequency
14,plate_age_id,0.009335,frequency
8,model,0.008444,frequency
13,trim,0.007902,frequency
9,generation,0.007300,frequency
7,make,0.007171,frequency
4,engine,0.006761,frequency
